## Read in cp3


In [1]:
import pandas as pd
import re
import os

In [2]:
INPUT_CSV = (
   "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint3_geocoded_2526.csv"
)

OUTPUT_CSV = (
    "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint4_cleaned_addresses.csv"
)
df = pd.read_csv(INPUT_CSV, low_memory=False)

print(f"Loaded: {INPUT_CSV}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

display(df.head())
with pd.option_context('display.max_rows', None):
    print(df['Location'].value_counts())




Loaded: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint3_geocoded_2526.csv
Rows: 66,741
Columns: 16


,Incident #,Date,Type,Location,Arrested,Location Prefix,Name,DOB,Charges,raw_address,cleaned_address,latitude,longitude,geocode_confidence,geocoded_at,coord_valid
0,25026764.0,2025-07-01 00:12:08,THREATS,259 ANDOVER ST,No,NaN,NaN,NaN,NaN,259 ANDOVER ST,"259 ANDOVER ST, Lawrence, MA",42.694842,-71.155948,10.0,2026-06-22T23:33:38.335777+00:00,True
1,25026765.0,2025-07-01 00:19:36,TOW/REPOSSED,454 HAMPSHIRE ST,No,NaN,NaN,NaN,NaN,454 HAMPSHIRE ST,"454 HAMPSHIRE ST, Lawrence, MA",42.717278,-71.171768,10.0,2026-06-25T23:39:24.046150+00:00,True
2,25026766.0,2025-07-01 00:49:07,DISTURBANCE,155 HAVERHILL ST,No,NaN,NaN,NaN,NaN,155 HAVERHILL ST,"155 HAVERHILL ST, Lawrence, MA",42.710380,-71.161736,10.0,2026-06-19T16:04:18.901970+00:00,True
3,25026767.0,2025-07-01 01:15:01,CK WELL BEING,621 COMMON ST,No,NaN,NaN,NaN,NaN,621 COMMON ST,"621 COMMON ST, Lawrence, MA",42.705034,-71.169622,10.0,2026-06-26T19:04:44.576031+00:00,True
4,25026769.0,2025-07-01 01:29:06,EXTRA SURVEIL,425 MERRIMACK ST,No,NaN,NaN,NaN,NaN,425 MERRIMACK ST,"425 MERRIMACK ST, Lawrence, MA",42.703240,-71.143607,10.0,2026-06-25T22:01:03.147567+00:00,True


Location
90 LOWELL ST                                 1038
50 BROADWAY                                   848
700 ESSEX ST                                  353
266 BROADWAY                                  336
296 ESSEX ST                                  327
73 WINTHROP AV                                309
370 BROADWAY                                  306
211 MERRIMACK ST                              273
BAY STATE RD                                  268
101 AMESBURY ST                               259
205 BROADWAY                                  257
164 MT VERNON ST                              247
BROADWAY & LOWELL ST                          242
550 BROADWAY                                  236
BROADWAY & ESSEX ST                           233
75 MANCHESTER ST                              222
240 CANAL ST                                  218
MARION AV                                     212
555 BROADWAY                                  208
MERRIMACK ST & S UNION ST                

## Clean up stuff we missed before the api calls

In [3]:
def clean_address_after_merge(address):
    """
    Clean the Location value after the geocoding merge.

    The function:
    - Converts text to uppercase
    - Removes periods, apostrophes, and backticks
    - Removes unwanted police-report prefixes/labels
    - Converts STREET to ST
    - Removes floor, apartment, and unit identifiers
    - Removes '#' and the characters attached to it until whitespace
    - Normalizes whitespace
    """
    if pd.isna(address):
        return ""

    address = str(address).upper().strip()

    # ---------------------------------------------------------
    # 1. Remove unwanted police-report terms
    # ---------------------------------------------------------
    unwanted_patterns = [
    r"\bREC\s*/\s*STOLEN\s+ARREST\b",
    r"\bARREST\s*/",
    r"\bV\s+TRUST\s*/",
    r"\bV\s+STOP\s*/",
    r"\bTOWED\b",
    r"\bWALK\s+IN\b",
    r"\bWALKIN\s*/",
    r"\bTOW\s*/",
    r"\bSTOP\s*/",
    r"\bV\s*/",
]

    for pattern in unwanted_patterns:
        address = re.sub(
            pattern,
            " ",
            address,
            flags=re.IGNORECASE,
        )
    
# Remove anything inside parentheses (including the parentheses)
# Examples:
# (REAR)
# (A)
# (PARKING LOT)
    address = re.sub(r"\([^)]*\)", " ", address)
    # Remove punctuation characters
# Examples:
# 150` ANDOVER ST
# 217 E HAVERHILL ST'
# 17 CYPRESS AVE.
# 23 BROADWAY°
    address = re.sub(r"[.'`°]", "", address)

    # ---------------------------------------------------------
    # 3. Standardize STREET to ST
    # ---------------------------------------------------------
    # 473 HAVERHILL STREET -> 473 HAVERHILL ST
    address = re.sub(
        r"\bSTREET\b",
        "ST",
        address,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 4. Remove floor identifiers
    # ---------------------------------------------------------
    # Handles:
    # 218 PARK ST FL 2
    # 300 CANAL ST FL3
    # 300 CANAL ST FLOOR 3
    #
    # Only removes the floor identifier, not everything after it.
    address = re.sub(
    r"\b(?:FLOOR\s*#?\s*[A-Z0-9-]+|FL\s*#?\s*\d+[A-Z0-9-]*)\b",
    " ",
    address,
    flags=re.IGNORECASE,
)

    # Remove FL or FLOOR when no value follows
   

    # ---------------------------------------------------------
    # 5. Remove apartment and unit identifiers
    # ---------------------------------------------------------
    # Handles:
    # APT 2
    # APT2
    # APARTMENT 4B
    # UNIT #5
    address = re.sub(
        r"\b(?:APT|APARTMENT|UNIT)\s*#?\s*[A-Z0-9-]+\b",
        " ",
        address,
        flags=re.IGNORECASE,
    )

    # Remove standalone APT, APARTMENT, or UNIT
    address = re.sub(
        r"\b(?:APT|APARTMENT|UNIT)\b",
        " ",
        address,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 6. Remove '#' and anything attached to it until whitespace
    # ---------------------------------------------------------
    # Examples:
    # 218 BROADWAY #1ST       -> 218 BROADWAY
    # 300 CANAL ST #311 FL3   -> 300 CANAL ST FL3
    #
    # FL3 is then removed by the floor rule above.
    address = re.sub(
        r"#\S*",
        " ",
        address,
    )

    # Remove any standalone number signs that remain
    address = address.replace("#", " ")

    # ---------------------------------------------------------
    # 7. Normalize whitespace
    # ---------------------------------------------------------
    address = re.sub(r"\s+", " ", address).strip()

    return address

## TEST

In [37]:
test_addresses = [
    "150` ANDOVER ST",
    "150` ARLINGTON ST",
    "17 CYPRESS AVE.",
    "217 E HAVERHILL ST'",
    "218 BROADWAY #1ST",
    "218 PARK ST FL 2",
    "300 CANAL ST #311 FL3",
    "473 HAVERHILL STREET",
    "V STOP / 250 BROADWAY",
    "STOP / 250 BROADWAY",
    "V / 250 BROADWAY",
    "V TRUST / 250 BROADWAY",
    "TOW/ 250 BROADWAY",
    "TOWED 250 BROADWAY",
    "WALK IN 250 BROADWAY",
    "REC/STOLEN ARREST 250 BROADWAY",
]

test_results = pd.DataFrame({
    "original": test_addresses,
    "cleaned": [
        clean_address_after_merge(address)
        for address in test_addresses
    ],
})

display(test_results)

,original,cleaned
0,150` ANDOVER ST,150 ANDOVER ST
1,150` ARLINGTON ST,150 ARLINGTON ST
2,17 CYPRESS AVE.,17 CYPRESS AVE
3,217 E HAVERHILL ST',217 E HAVERHILL ST
4,218 BROADWAY #1ST,218 BROADWAY
5,218 PARK ST FL 2,218 PARK ST
6,300 CANAL ST #311 FL3,300 CANAL ST
7,473 HAVERHILL STREET,473 HAVERHILL ST
8,V STOP / 250 BROADWAY,250 BROADWAY
9,STOP / 250 BROADWAY,250 BROADWAY


In [4]:
cleaned_df = df.copy()

cleaned_df["cleaned_location"] = (
    cleaned_df["Location"]
    .apply(clean_address_after_merge)
)

print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning: {len(cleaned_df):,}")

Rows before cleaning: 66,741
Rows after cleaning: 66,741


## Compare and consolidate addresses 

In [5]:
comparison = cleaned_df[
    ["Location", "cleaned_location"]
].drop_duplicates()

changed = comparison[
    comparison["Location"].fillna("").astype(str).str.strip()
    != comparison["cleaned_location"]
]

print(f"Unique original locations: {cleaned_df['Location'].nunique(dropna=True):,}")
print(f"Unique cleaned locations: {cleaned_df['cleaned_location'].nunique(dropna=True):,}")
print(f"Unique values changed by cleaning: {len(changed):,}")

display(changed.head(50))

Unique original locations: 17,592
Unique cleaned locations: 13,407
Unique values changed by cleaning: 5,050


,Location,cleaned_location
50,25 MARSTON ST #303,25 MARSTON ST
51,148 BUTLER ST FL 1,148 BUTLER ST
52,12 MAGINNIS AV #BUILDINGS- APT4,12 MAGINNIS AV
69,318 MARKET ST FL 3,318 MARKET ST
83,74 UNION ST #31,74 UNION ST
89,56-58 KENT ST FL 1,56-58 KENT ST
90,40 LAWRENCE ST #313,40 LAWRENCE ST
112,300 METHUEN ST #427,300 METHUEN ST
117,134 FOSTER ST FL 1,134 FOSTER ST
124,266 BRROADWAY STREET,266 BRROADWAY ST


In [6]:
collapsed = (
    cleaned_df.loc[
        cleaned_df["cleaned_location"].ne(""),
        ["cleaned_location", "Location"],
    ]
    .drop_duplicates()
    .groupby("cleaned_location")["Location"]
    .agg(list)
    .reset_index()
)

collapsed = collapsed[
    collapsed["Location"].apply(len) > 1
].copy()

collapsed["original_address_count"] = (
    collapsed["Location"].apply(len)
)

collapsed = collapsed.sort_values(
    "original_address_count",
    ascending=False,
)

print(f"Collapsed address groups: {len(collapsed):,}")

display(collapsed.head(50))

Collapsed address groups: 1,974


,cleaned_location,Location,original_address_count
5759,300 METHUEN ST,"[300 METHUEN ST #427, 300 METHUEN ST #406, 300...",95
4572,240 CANAL ST,"[240 CANAL ST, 240 CANAL ST #536 FL 5, 240 CAN...",55
4774,250 MERRIMACK ST,"[250 MERRIMACK ST, 250 MERRIMACK ST #274, 250 ...",50
6007,32 LAWRENCE ST,"[32 LAWRENCE ST #39, 32 LAWRENCE ST #48, 32 LA...",44
5744,300 CANAL ST,"[300 CANAL ST, 300 CANAL ST #221, 300 CANAL ST...",43
780,11 LAWRENCE ST,"[11 LAWRENCE ST, 11 LAWRENCE ST #2 FL 2, 11 LA...",41
10599,77 S UNION ST,"[77 S UNION ST, 77 S UNION ST #308, 77 S UNION...",37
4651,248 BROADWAY,"[248 BROADWAY, 248 BROADWAY #215, 248 BROADWAY...",36
3578,2 MUSEUM SQ,"[2 MUSEUM SQ, 2 MUSEUM SQ #216, 2 MUSEUM SQ #4...",35
9260,590 BROADWAY,"[590 BROADWAY, 590 BROADWAY #206, 590 BROADWAY...",33


In [9]:
with pd.option_context('display.max_rows', None):
    print(cleaned_df['Location'].value_counts())
coordinate_columns = [
    column
    for column in ["latitude", "longitude", "lat", "long"]
    if column in cleaned_df.columns
]

print("Coordinate columns found:", coordinate_columns)

if coordinate_columns:
    print(
        cleaned_df[coordinate_columns]
        .notna()
        .sum()
    )
display(cleaned_df.head())

Location
90 LOWELL ST                                 1038
50 BROADWAY                                   848
700 ESSEX ST                                  353
266 BROADWAY                                  336
296 ESSEX ST                                  327
73 WINTHROP AV                                309
370 BROADWAY                                  306
211 MERRIMACK ST                              273
BAY STATE RD                                  268
101 AMESBURY ST                               259
205 BROADWAY                                  257
164 MT VERNON ST                              247
BROADWAY & LOWELL ST                          242
550 BROADWAY                                  236
BROADWAY & ESSEX ST                           233
75 MANCHESTER ST                              222
240 CANAL ST                                  218
MARION AV                                     212
555 BROADWAY                                  208
MERRIMACK ST & S UNION ST                

,Incident #,Date,Type,Location,Arrested,Location Prefix,Name,DOB,Charges,raw_address,cleaned_address,latitude,longitude,geocode_confidence,geocoded_at,coord_valid,cleaned_location
0,25026764.0,2025-07-01 00:12:08,THREATS,259 ANDOVER ST,No,NaN,NaN,NaN,NaN,259 ANDOVER ST,"259 ANDOVER ST, Lawrence, MA",42.694842,-71.155948,10.0,2026-06-22T23:33:38.335777+00:00,True,259 ANDOVER ST
1,25026765.0,2025-07-01 00:19:36,TOW/REPOSSED,454 HAMPSHIRE ST,No,NaN,NaN,NaN,NaN,454 HAMPSHIRE ST,"454 HAMPSHIRE ST, Lawrence, MA",42.717278,-71.171768,10.0,2026-06-25T23:39:24.046150+00:00,True,454 HAMPSHIRE ST
2,25026766.0,2025-07-01 00:49:07,DISTURBANCE,155 HAVERHILL ST,No,NaN,NaN,NaN,NaN,155 HAVERHILL ST,"155 HAVERHILL ST, Lawrence, MA",42.710380,-71.161736,10.0,2026-06-19T16:04:18.901970+00:00,True,155 HAVERHILL ST
3,25026767.0,2025-07-01 01:15:01,CK WELL BEING,621 COMMON ST,No,NaN,NaN,NaN,NaN,621 COMMON ST,"621 COMMON ST, Lawrence, MA",42.705034,-71.169622,10.0,2026-06-26T19:04:44.576031+00:00,True,621 COMMON ST
4,25026769.0,2025-07-01 01:29:06,EXTRA SURVEIL,425 MERRIMACK ST,No,NaN,NaN,NaN,NaN,425 MERRIMACK ST,"425 MERRIMACK ST, Lawrence, MA",42.703240,-71.143607,10.0,2026-06-25T22:01:03.147567+00:00,True,425 MERRIMACK ST


In [10]:
cleaned_df = cleaned_df.drop(columns=["Source PDF"], errors="ignore")

In [11]:
# Replace Location with the cleaned version
cleaned_df["Location"] = cleaned_df["cleaned_location"]

# Remove the temporary column
cleaned_df = cleaned_df.drop(columns=["cleaned_location"])

## Create cp4

In [12]:
os.makedirs(
    os.path.dirname(OUTPUT_CSV),
    exist_ok=True,
)

cleaned_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print(f"Saved: {OUTPUT_CSV}")
print(f"Saved rows: {len(cleaned_df):,}")

Saved: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint4_cleaned_addresses.csv
Saved rows: 66,741


## Verify 

In [13]:
verification_df = pd.read_csv(
    OUTPUT_CSV,
    low_memory=False,
)

print(f"Reloaded rows: {len(verification_df):,}")
print(f"Reloaded columns: {len(verification_df.columns):,}")
print(
    "cleaned_location present:",
    "cleaned_location" in verification_df.columns,
)

display(
    verification_df[
        ["Location"]
    ].head(20)
)

Reloaded rows: 66,741
Reloaded columns: 16
cleaned_location present: False


,Location
0,259 ANDOVER ST
1,454 HAMPSHIRE ST
2,155 HAVERHILL ST
3,621 COMMON ST
4,425 MERRIMACK ST
5,EMBANKMENT RD
6,BAY STATE RD
7,GREENFIELD ST
8,CONDUIT ST
9,MARKET ST & S UNION ST
